# House Price Prediction -  ML Project
### Dataset: Housing.csv (545 rows × 13 features)
---
**Features:** area, bedrooms, bathrooms, stories, mainroad, guestroom, basement, hotwaterheating, airconditioning, parking, prefarea, furnishingstatus  
**Target:** price

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

## Step 1: Load Dataset

In [ ]:
df = pd.read_csv('Dataset/Housing.csv')

print(f"Shape: {df.shape}")
print(f"Missing Values: {df.isnull().sum().sum()}")
df.head()

## Step 2: Exploratory Data Analysis (EDA)

In [ ]:
df.describe()

In [ ]:
# Data types and info
df.info()

In [ ]:
# Visualization: Price Distribution + Correlation
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('EDA Overview', fontsize=16, fontweight='bold')

# Price Distribution
axes[0].hist(df['price'] / 1e6, bins=30, color='#4F46E5', edgecolor='white', alpha=0.85)
axes[0].set_title('Price Distribution')
axes[0].set_xlabel('Price (Millions)')
axes[0].set_ylabel('Count')
axes[0].axvline(df['price'].mean() / 1e6, color='red', linestyle='--', linewidth=2, label=f"Mean: {df['price'].mean()/1e6:.1f}M")
axes[0].legend()

# Boxplot: Price by Bedrooms
df.boxplot(column='price', by='bedrooms', ax=axes[1])
axes[1].set_title('Price by Bedrooms')
axes[1].set_xlabel('Bedrooms')
axes[1].set_ylabel('Price')
plt.suptitle('')

plt.tight_layout()
plt.show()

In [ ]:
# Price vs Area scatter
plt.figure(figsize=(8, 5))
sc = plt.scatter(df['area'], df['price'] / 1e6, c=df['bedrooms'], cmap='viridis', alpha=0.65, s=30)
plt.colorbar(sc, label='Bedrooms')
plt.title('Price vs Area', fontsize=14, fontweight='bold')
plt.xlabel('Area (sq ft)')
plt.ylabel('Price (Millions)')
plt.tight_layout()
plt.show()

##  Step 3: Preprocessing

In [ ]:
# Encode binary yes/no columns
binary_cols = ['mainroad', 'guestroom', 'basement',
               'hotwaterheating', 'airconditioning', 'prefarea']
for col in binary_cols:
    df[col] = df[col].map({'yes': 1, 'no': 0})

# Encode furnishingstatus (ordinal)
furnish_map = {'furnished': 2, 'semi-furnished': 1, 'unfurnished': 0}
df['furnishingstatus'] = df['furnishingstatus'].map(furnish_map)

print("Encoding complete!")
df.head()

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(10, 7))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm',
            square=True, linewidths=0.5)
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

##  Step 4: Train / Test Split

In [ ]:
X = df.drop('price', axis=1)
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale features (for Linear Regression)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Train set : {X_train.shape}")
print(f"Test set  : {X_test.shape}")

## Step 5: Train ML Models

In [ ]:
models = {
    'Linear Regression':  LinearRegression(),
    'Random Forest':      RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting':  GradientBoostingRegressor(n_estimators=100, random_state=42),
}

results = {}

for name, model in models.items():
    X_tr = X_train_sc if name == 'Linear Regression' else X_train
    X_te = X_test_sc  if name == 'Linear Regression' else X_test

    model.fit(X_tr, y_train)
    y_pred = model.predict(X_te)

    r2   = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae  = mean_absolute_error(y_test, y_pred)

    results[name] = {'model': model, 'y_pred': y_pred,
                     'r2': r2, 'rmse': rmse, 'mae': mae}

    print(f"{name:<28} | R²: {r2:.4f} | RMSE: {rmse:,.0f} | MAE: {mae:,.0f}")

## Step 6: Model Evaluation & Comparison

In [ ]:
# Summary Table
results_df = pd.DataFrame({
    'Model': list(results.keys()),
    'R² Score': [results[m]['r2'] for m in results],
    'RMSE': [results[m]['rmse'] for m in results],
    'MAE':  [results[m]['mae']  for m in results],
})
results_df = results_df.sort_values('R² Score', ascending=False).reset_index(drop=True)
results_df

In [ ]:
# Bar chart comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Model Performance Comparison', fontsize=15, fontweight='bold')

metrics = ['R² Score', 'RMSE', 'MAE']
palette = ['#4F46E5', '#7C3AED', '#06B6D4']

for i, metric in enumerate(metrics):
    bars = axes[i].bar(results_df['Model'], results_df[metric], color=palette, edgecolor='white', width=0.5)
    axes[i].set_title(metric, fontweight='bold')
    axes[i].set_xticklabels(results_df['Model'], rotation=15, ha='right', fontsize=9)
    for bar, val in zip(bars, results_df[metric]):
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.01,
                     f'{val:,.0f}' if metric != 'R² Score' else f'{val:.3f}',
                     ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

## Step 7: Best Model Analysis

In [ ]:
best_name = results_df.iloc[0]['Model']
best = results[best_name]
best_model = best['model']
y_pred_best = best['y_pred']

print(f" Best Model : {best_name}")
print(f"   R² Score   : {best['r2']:.4f}")
print(f"   RMSE       : {best['rmse']:,.0f}")
print(f"   MAE        : {best['mae']:,.0f}")

# Cross Validation
X_cv = X_train_sc if best_name == 'Linear Regression' else X_train
cv_scores = cross_val_score(best_model, X_cv, y_train, cv=5, scoring='r2')
print(f"\n   5-Fold CV R² : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

In [ ]:
# Actual vs Predicted + Residuals
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(f'Best Model: {best_name}', fontsize=14, fontweight='bold')

# Actual vs Predicted
axes[0].scatter(y_test / 1e6, y_pred_best / 1e6, alpha=0.6, color='#7C3AED', s=30)
mn = min(y_test.min(), y_pred_best.min()) / 1e6
mx = max(y_test.max(), y_pred_best.max()) / 1e6
axes[0].plot([mn, mx], [mn, mx], 'r--', linewidth=2, label='Perfect Fit')
axes[0].set_title('Actual vs Predicted')
axes[0].set_xlabel('Actual Price (M)')
axes[0].set_ylabel('Predicted Price (M)')
axes[0].legend()

# Residuals
residuals = y_test - y_pred_best
axes[1].scatter(y_pred_best / 1e6, residuals / 1e6, alpha=0.6, color='#F59E0B', s=30)
axes[1].axhline(0, color='red', linewidth=2, linestyle='--')
axes[1].set_title('Residual Plot')
axes[1].set_xlabel('Predicted Price (M)')
axes[1].set_ylabel('Residuals (M)')

plt.tight_layout()
plt.show()

## Step 8: Feature Importance

In [ ]:
rf_model  = results['Random Forest']['model']
feat_imp  = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(10, 5))
colors = ['#06B6D4' if i == 0 else '#4F46E5' for i in range(len(feat_imp))]
feat_imp.plot(kind='bar', color=colors, edgecolor='white')
plt.title('Feature Importance (Random Forest)', fontsize=14, fontweight='bold')
plt.xlabel('')
plt.ylabel('Importance Score')
plt.xticks(rotation=35)
for i, v in enumerate(feat_imp):
    plt.text(i, v + 0.002, f'{v:.3f}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

print("\n Top 3 Most Important Features:")
for i, (feat, val) in enumerate(feat_imp.head(3).items(), 1):
    print(f"   {i}. {feat}: {val:.4f}")

## Step 9: Predict New House Price

In [ ]:
def predict_price(area, bedrooms, bathrooms, stories,
                  mainroad='yes', guestroom='no', basement='no',
                  hotwaterheating='no', airconditioning='yes',
                  parking=1, prefarea='no', furnishingstatus='semi-furnished'):
    """Predict house price using the best trained model."""
    sample = pd.DataFrame([{
        'area': area, 'bedrooms': bedrooms, 'bathrooms': bathrooms,
        'stories': stories,
        'mainroad': 1 if mainroad == 'yes' else 0,
        'guestroom': 1 if guestroom == 'yes' else 0,
        'basement': 1 if basement == 'yes' else 0,
        'hotwaterheating': 1 if hotwaterheating == 'yes' else 0,
        'airconditioning': 1 if airconditioning == 'yes' else 0,
        'parking': parking,
        'prefarea': 1 if prefarea == 'yes' else 0,
        'furnishingstatus': {'furnished': 2, 'semi-furnished': 1, 'unfurnished': 0}.get(furnishingstatus, 1),
    }])
    if best_name == 'Linear Regression':
        price = best_model.predict(scaler.transform(sample))[0]
    else:
        price = best_model.predict(sample)[0]
    return price

# ── Try it out ──────────────────────────────────────────────
price = predict_price(
    area=5000, bedrooms=3, bathrooms=2, stories=2,
    mainroad='yes', airconditioning='yes', parking=1,
    furnishingstatus='semi-furnished'
)
print(f"🏠 Input  : 5000 sqft | 3 bed | 2 bath | 2 story | AC: yes")
print(f"💰 Predicted Price → {price:,.0f}  ({price/1e6:.2f} Million)")